In [ ]:
import pandas as pd
import duckdb
from pathlib import Path
import unicodedata
import re
import json
from sentence_transformers import SentenceTransformer
from dataclasses import dataclass, field
from rapidfuzz import fuzz, process

c:\Users\ACER\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sheets = pd.read_excel('input.xlsx', sheet_name=None)
master_df = sheets[list(sheets.keys())[0]]
hire_team_df = sheets[list(sheets.keys())[1]]
new_employee_df = sheets[list(sheets.keys())[2]]
employee_list_df = sheets[list(sheets.keys())[3]]

In [3]:
"""
Xử lý 5 cột output đầu tiên. Lấy dữ liệu từ employee_list_df
"""
result = employee_list_df[['Employee Local ID']].copy()
result['WorkDay ID'] = employee_list_df[['WorkDay ID']].copy()
result['First Name'] = employee_list_df[['First Name']].copy()
result['Date Of Birth'] = employee_list_df[['Date Of Birth']].copy()
result['Service Start Date'] = employee_list_df[['Service Start Date']].copy()
# Format lại cột ['Date Of Birth'] và ['Service Start Date']
result['Date Of Birth'] = pd.to_datetime(result['Date Of Birth']).dt.strftime('%d-%m-%Y')
result['Service Start Date'] = pd.to_datetime(result['Service Start Date']).dt.strftime('%d-%m-%Y')

In [4]:
"""New Hires Reason"""
lookup = duckdb.query("""
    SELECT
        el."WorkDay ID",
        ht."Headcount" AS "New Hires Reason"
    FROM employee_list_df el
    LEFT JOIN hire_team_df ht
        ON el."WorkDay ID" = ht."Employee ID"
""").to_df()

result = result.merge(lookup, on="WorkDay ID", how="left")

In [5]:
"""Organization's name"""
lookup = duckdb.query("""
    SELECT
        el."WorkDay ID",
        m."[02-BU/CF]" AS "[02-BU/CF]",
        m."[03-SBU/SCF]" AS "[03-SBU/SCF]",
        m."[04-Section]" AS "[04-Section]",
        m."[05-Team]" AS "[05-Team]",
        m."[06-Sub-Team]" AS "[06-Sub-Team]",
        m."[07-Unit]" AS "[07-Unit]",
        m."[08-Sub-Unit]" AS "[08-Sub-Unit]",
        m."[09-Sub - sub unit]" AS "[09-Sub - sub unit]"
    FROM employee_list_df el
    LEFT JOIN new_employee_df ne
        ON TRIM(CAST(el."WorkDay ID" AS VARCHAR)) = TRIM(CAST(ne."WD ID" AS VARCHAR))
    LEFT JOIN master_df m
        ON TRIM(CAST(ne."ID same team" AS VARCHAR)) = TRIM(CAST(m."Employee Local ID" AS VARCHAR))
""").to_df()

result = result.merge(lookup, on="WorkDay ID", how="left")

In [6]:
"""Địa chỉ đường"""
lookup = duckdb.query("""
    SELECT 
        el."WorkDay ID",
        ne."Thành phố (Province/City)" AS "Tỉnh/Thành phố",
        ne."Phường/ Xã (Ward)" AS "Xã/Phường",
        ne."Số nhà & Tên đường (No. & Street)" AS "Địa chỉ đường",
        ne."Số tài khoản ngân hàng (Bank Account No.)" AS "Số tài khoản",
        ne."Tên ngân hàng (Bank Name)" AS "Tên ngân hàng",
        ne."Chi nhánh (Bank Branch)" AS "Chi nhánh ngân hàng"
    FROM employee_list_df el
    LEFT JOIN new_employee_df ne
        ON TRIM(CAST(el."WorkDay ID" AS VARCHAR)) = TRIM(CAST(ne."WD ID" AS VARCHAR))                                      
""").to_df()
result = result.merge(lookup, on="WorkDay ID", how="left")

In [7]:
result['Tên chủ tài khoản'] = result[['First Name']].copy()
def remove_diacritics(text):
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = text.replace('đ', 'd').replace('Đ', 'D')  # đ isn't handled by NFD decomposition
    return text
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
result['Tên chủ tài khoản'] = result['Tên chủ tài khoản'].apply(lambda x: remove_diacritics(x).upper() if pd.notna(x) else x)
result['Tên ngân hàng'] = result['Tên ngân hàng'].apply(lambda x: remove_diacritics(x) if pd.notna(x) else x)
result['Tỉnh/Thành phố'] = result['Tỉnh/Thành phố'].apply(lambda x: remove_diacritics(x) if pd.notna(x) else x)
result['Xã/Phường'] = result['Xã/Phường'].apply(lambda x: remove_diacritics(x) if pd.notna(x) else x)
result['Chi nhánh ngân hàng'] = result['Chi nhánh ngân hàng'].apply(lambda x: remove_diacritics(x) if pd.notna(x) else x)

In [8]:
import importlib
import setup

importlib.reload(setup)

from setup import banks, bank_aliases
from setup import provinces, province_old_to_new_alias
from setup import wards, ward_old_to_new_alias
from setup import banks, bank_aliases
from setup import provinces, province_old_to_new_alias
from setup import wards, ward_old_to_new_alias

print(len(banks))
print(len(provinces))
print(len(wards))

37
34
3321


In [9]:
def normalize(s):
    """
     # Uppercase / lowercase
    ("TECHCOMBANK", "techcombank"),
    ("TechComBank", "techcombank"),
    ("tEcHcOmBaNk", "techcombank"),

    # Special characters
    ("MUFG BANK,LTD.", "mufg bank ltd"),
    ("Bank of China!!!", "bank of china"),
    ("Cake@VPBank", "cake vpbank"),
    """
    s = remove_diacritics(str(s)).lower()
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s

@dataclass
class ColumnConfig:
    name: str                          # tên cột trong dataframe, ví dụ "Tên ngân hàng"
    canonical_list: list                # danh sách giá trị chuẩn
    alias_dict: dict = field(default_factory=dict)  # canonical -> [aliases]
    threshold_high: int = 80            # ngưỡng tier2
    threshold_fallback: int = 65        # ngưỡng tier3
    llm_batch_size: int = 15

    def build_lookup(self):
        """
        canonical_list = [
            "Techcombank",
            "Vietcombank",
            "ACB"
        ]

        alias_dict = {
            "Techcombank": ["TCB", "Ngân hàng Kỹ Thương", "NHTMCP Kỹ Thương VN"],
            "Vietcombank": ["VCB", "Ngân hàng Ngoại Thương"],
        }

        --> lookup = 
        {
            "techcombank": "Techcombank",
            "vietcombank": "Vietcombank",
            "acb": "ACB",

            "tcb": "Techcombank",
            "ngan hang ky thuong": "Techcombank",
            "nhtmcp ky thuong vn": "Techcombank",

            "vcb": "Vietcombank",
            "ngan hang ngoai thuong": "Vietcombank",
        }
        """
        lookup = {}

        for canonical in self.canonical_list:
            lookup[normalize(canonical)] = canonical
        
        for canonical, aliases in self.alias_dict.items():
            for a in aliases:
                lookup[normalize(a)] = canonical
        
        return lookup

class ClosedSetMatcher:
    """Chạy 5-tier pipeline cho MỘT cột, dùng chung logic cho mọi cột."""

    def __init__(self, config: ColumnConfig, llm_model=None):
        self.config = config
        self.lookup = config.build_lookup()
        self.lookup_keys = list(self.lookup.keys())
        self.llm_model = llm_model

    # Tier 1: rule-based exact match
    def _rule_based(self, value):
        q = normalize(value)
        if q in self.lookup:
            return self.lookup[q], 100, "tier1_rule_based" #100 = confidence sc
        return None, 0, None
    
    # Tier 2: rapid-fuzz high confidence
    def _rapidfuzz_high(self, value):
        q = normalize(value)

        if not q:
            return None, 0, None

        best_match = None
        best_score = 0

        scorers = [
            fuzz.token_sort_ratio,
            fuzz.token_set_ratio,
            fuzz.WRatio,
        ]

        for scorer in scorers:
            match, score, _ = process.extractOne(
                q,
                self.lookup_keys,
                scorer=scorer
            )

            if score > best_score:
                best_score = score
                best_match = match

        if best_score >= self.config.threshold_high:
            return self.lookup[best_match], best_score, "tier2_rapidfuzz_high"

        return None, best_score, None
    
    # Tier 3: rapid-fuzz fallback, low confidence
    def _rapidfuzz_fallback(self, value):
        q = normalize(value)
        if not q:
            return None, 0, None
        match, score, _ = process.extractOne(q, self.lookup_keys, scorer=fuzz.token_set_ratio)
        if score >= self.config.threshold_fallback:
            return self.lookup[match], score, "tier3_rapidfuzz_fallback"
        return None, score, None
    
    # Tier 4: LLM classification cho phần còn sót
    def _llm_classify(self, values):
        if not self.llm_model or not values:
            return {}
        canonical_list = self.config.canonical_list
        results = {}
        bs = self.config.llm_batch_size
        for i in range(0, len(values), bs):
            batch = values[i:i+bs]
            prompt = f"""
                Bạn là chuyên gia chuẩn hóa dữ liệu hành chính/ngân hàng Việt Nam cho cột "{self.config.name}".
                Danh sách giá trị CHUẨN (closed-set, chỉ chọn trong đây):
                {json.dumps(canonical_list, ensure_ascii=False, indent=2)}

                Map mỗi giá trị "bẩn" sau về đúng 1 giá trị chuẩn. Nếu không chắc chắn, trả "UNKNOWN".
                Giá trị cần phân loại:
                {json.dumps(batch, ensure_ascii=False, indent=2)}

                Chỉ trả JSON {{"giá trị bẩn": "giá trị chuẩn hoặc UNKNOWN"}}, không giải thích, không markdown.
            """
            response = self.llm_model.generate_content(prompt)
            text = response.text.strip().replace("```json", "").replace("```", "")
            try:
                results.update(json.loads(text))
            except json.JSONDecodeError:
                for v in batch:
                    results[v] = "PARSE_ERROR"
        return results
    
    def run(self, values: list) -> pd.DataFrame:
        records = [{"raw": v, "matched": None, "score": 0, "tier": None} for v in values]

        remaining = []
        for idx, rec in enumerate(records):
            m, s, t = self._rule_based(rec["raw"])
            if m: rec.update(matched=m, score=s, tier=t)
            else: remaining.append(idx)

        still = []
        for idx in remaining:
            m, s, t = self._rapidfuzz_high(records[idx]["raw"])
            if m: records[idx].update(matched=m, score=s, tier=t)
            else: still.append(idx)

        still2 = []
        for idx in still:
            m, s, t = self._rapidfuzz_fallback(records[idx]["raw"])
            if m: records[idx].update(matched=m, score=s, tier=t)
            else: still2.append(idx)

        if still2:
            names_for_llm = [records[idx]["raw"] for idx in still2]
            llm_results = self._llm_classify(names_for_llm)
            for idx in still2:
                answer = llm_results.get(records[idx]["raw"], "UNKNOWN")
                if answer in self.config.canonical_list:
                    records[idx].update(matched=answer, score=None, tier="tier4_llm")
                else:
                    records[idx].update(matched=None, score=None, tier="tier5_review_tay")

        return pd.DataFrame(records)

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
llm_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

bank_config = ColumnConfig(
    name="Tên ngân hàng",
    canonical_list=banks,
    alias_dict=bank_aliases,
    threshold_high=85,
    threshold_fallback=65,
)

province_config = ColumnConfig(
    name="Tỉnh/Thành phố",
    canonical_list=provinces,
    alias_dict=province_old_to_new_alias,
    threshold_high=85,
    threshold_fallback=65,
)

ward_config = ColumnConfig(
    name="Xã/Phường",
    canonical_list=wards,
    alias_dict=ward_old_to_new_alias,
    threshold_high=88,
    threshold_fallback=70,
)
"""
# ==== 4. Chi nhánh ngân hàng — không phải closed-set chặt, threshold nên nới ====
branch_config = ColumnConfig(
    name="Chi nhánh ngân hàng",
    canonical_list=branch_list,           # nếu có danh sách chi nhánh chính thức của từng bank
    alias_dict={},
    threshold_high=85,
    threshold_fallback=60,
)
"""

# ==== Chạy đồng loạt ====
#configs = [bank_config, province_config, ward_config, branch_config]

configs = [bank_config, province_config, ward_config]
final_results = {}

for cfg in configs:
    col_values = result[cfg.name].fillna("").astype(str).tolist()
    matcher = ClosedSetMatcher(cfg, llm_model=llm_model)
    df_match = matcher.run(col_values)
    final_results[cfg.name] = df_match

    print(f"\n=== {cfg.name} ===")
    print(df_match["tier"].value_counts())

    result[f"{cfg.name} (chuẩn hóa)"] = df_match["matched"].values
    result[f"{cfg.name} (tier)"] = df_match["tier"].values

# Gộp toàn bộ dòng cần review tay của cả 4 cột vào 1 file để xử lý 1 lần
review_frames = []
for col_name, df_match in final_results.items():
    sub = df_match[df_match["tier"] == "tier5_review_tay"].copy()
    sub["column"] = col_name
    review_frames.append(sub)

print(pd.concat(review_frames, ignore_index=True))


=== Tên ngân hàng ===
tier
tier1_rule_based        15
tier2_rapidfuzz_high     3
Name: count, dtype: int64

=== Tỉnh/Thành phố ===
tier
tier1_rule_based        17
tier2_rapidfuzz_high     1
Name: count, dtype: int64

=== Xã/Phường ===
tier
tier1_rule_based        17
tier2_rapidfuzz_high     1
Name: count, dtype: int64
Empty DataFrame
Columns: [raw, matched, score, tier, column]
Index: []


In [11]:
cols_to_check = []
for cfg in configs:
    cols_to_check += [cfg.name, f"{cfg.name} (chuẩn hóa)", f"{cfg.name} (tier)"]

In [13]:
output_path = Path("output.xlsx")
output_path.unlink(missing_ok=True)

from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter
import re

new_order = [
    "Employee Local ID",
    "WorkDay ID",
    "First Name",
    "Date Of Birth",
    "Service Start Date",
    "New Hires Reason",
    "[02-BU/CF]",
    "[03-SBU/SCF]",
    "[04-Section]",
    "[05-Team]",
    "[06-Sub-Team]",
    "[07-Unit]",
    "[08-Sub-Unit]",
    "[09-Sub - sub unit]",
    "Tỉnh/Thành phố",
    "Tỉnh/Thành phố (chuẩn hóa)",
    "Tỉnh/Thành phố (tier)",
    "Xã/Phường",
    "Xã/Phường (chuẩn hóa)",
    "Xã/Phường (tier)",
    "Địa chỉ đường",
    "Tên chủ tài khoản",
    "Tên ngân hàng",
    "Tên ngân hàng (chuẩn hóa)",
    "Tên ngân hàng (tier)",
    "Chi nhánh ngân hàng",
    "Số tài khoản",
]

# Giữ bản có cột tier để dùng cho logic tô màu
result_with_tier = result[new_order].copy()

# File output cuối cùng sẽ bỏ toàn bộ cột tier
output_cols = [col for col in new_order if not col.endswith(" (tier)")]
output_df = result_with_tier[output_cols].copy()
output_df.to_excel(output_path, index=False)

wb = load_workbook(output_path)
ws = wb.worksheets[0]

# Màu đỏ nhạt
red_fill = PatternFill(
    start_color="FFC7CE",
    end_color="FFC7CE",
    fill_type="solid"
)

# Map tên cột -> vị trí cột trong file Excel sau khi đã bỏ cột tier
header_to_col = {
    cell.value: cell.column
    for cell in ws[1]
}

def is_tier2_or_below(tier_value):
    """
    tier1: không tô
    tier2, tier3, tier4, tier5: tô đỏ
    """
    if pd.isna(tier_value):
        return False

    match = re.search(r"tier(\d+)", str(tier_value).lower())
    return bool(match and int(match.group(1)) >= 2)

for cfg in configs:
    raw_col = cfg.name
    normalized_col = f"{cfg.name} (chuẩn hóa)"
    tier_col = f"{cfg.name} (tier)"

    # Bỏ qua nếu thiếu cột để tránh lỗi khi đổi config
    if tier_col not in result_with_tier.columns:
        continue
    if raw_col not in header_to_col or normalized_col not in header_to_col:
        continue

    raw_excel_col = header_to_col[raw_col]
    normalized_excel_col = header_to_col[normalized_col]

    for row_idx, tier_value in enumerate(result_with_tier[tier_col], start=2):
        if is_tier2_or_below(tier_value):
            ws.cell(row=row_idx, column=raw_excel_col).fill = red_fill
            ws.cell(row=row_idx, column=normalized_excel_col).fill = red_fill

# Auto fit độ rộng cột
for col_idx, col_name in enumerate(output_df.columns, start=1):
    max_len = max(
        output_df[col_name].astype(str).map(len).max(),
        len(str(col_name))
    )
    col_letter = get_column_letter(col_idx)
    ws.column_dimensions[col_letter].width = max_len + 2

wb.save(output_path)